<a href="https://colab.research.google.com/github/Pranq-lgtm/TF-Scalable-Classifier/blob/main/tests_test_data_loader_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import pytest
import numpy as np
from src.data_loader import create_dataset # Assuming this is your function

def test_dataset_output_shapes():
    """
    Test that the dataset yields batches of the correct shape.
    Essential for ensuring the model input layer won't crash.
    """
    batch_size = 32
    img_height, img_width = 224, 224

    # Mock dataset or call your loader with a small sample
    ds = create_dataset(batch_size=batch_size, img_shape=(img_height, img_width))

    # Get one batch
    features, labels = next(iter(ds.take(1)))

    assert features.shape == (batch_size, img_height, img_width, 3)
    assert labels.shape == (batch_size,)

def test_dataset_normalization():
    """
    Test that image pixels are normalized between [0, 1].
    Prevents training divergence issues.
    """
    ds = create_dataset(batch_size=1)
    features, _ = next(iter(ds.take(1)))

    assert tf.reduce_max(features) <= 1.0
    assert tf.reduce_min(features) >= 0.0

def test_dataset_is_shuffled():
    """
    Validation for data integrity: ensures we aren't seeing
    the exact same order every time (if shuffling is enabled).
    """
    ds1 = create_dataset(batch_size=5, shuffle=True, seed=42)
    ds2 = create_dataset(batch_size=5, shuffle=True, seed=7) # Different seed

    data1 = next(iter(ds1))
    data2 = next(iter(ds2))

    # Assert that the batches are not identical
    assert not np.array_equal(data1[0].numpy(), data2[0].numpy())